# 범용 손그림 탐지 모델 학습 (Stage A) — Quick Draw

**산출물 1: 범용 손그림(line-art) 객체탐지 모델.** Quick Draw 스케치를 랜덤 배치한
합성 장면으로 YOLO를 학습해 `htp_yolo/<SKETCH_MODEL_NAME>` 을 만든다.

이 노트북은 **HTP 데이터셋이 없어도 단독으로 돌아간다.** HTP 파인튜닝은
[htp_finetune.ipynb](htp_finetune.ipynb)가 담당하며, 이 노트북의 산출물 `.pt` 파일
하나만 입력으로 받는다. 즉 의존은 **파일 경유 단방향**이다.

```
sketch_base.ipynb ──> sketch_base*.pt ──> htp_finetune.ipynb ──> htp_best*.pt
```

### 이 노트북이 절대 건드리지 않는 것
`htp_best.pt`, `stageB_htp/`, `classes.json`. 클래스를 바꿔가며 몇 번을 재실행해도
HTP 산출물은 안전하다. (분리 전 단일 노트북에서는 Run All이 `htp_best.pt`를 덮어썼다.)

### 클래스를 확장하려면
`Config`의 `QD_CATEGORIES`에 카테고리를 추가하고 `QD_TRAIN_IMAGES`를 함께 올린다.
클래스만 늘리고 이미지 수를 그대로 두면 희귀 클래스가 학습되지 않는다.
**그리고 `SKETCH_MODEL_NAME`을 새 이름으로 바꿔** 기존 가중치를 보존한다 —
`htp_best.pt`는 기존 `sketch_base.pt`에서 파생된 산출물이라 그 조상을 지우면 계보가 끊긴다.

### ⚠️ 캐시 무효화 규칙
| 바꾼 것 | 지워야 반영되는 것 |
|---|---|
| `QD_CATEGORIES`, `QD_TRAIN_IMAGES`, `QD_CANVAS` 등 합성 파라미터 | `htp_yolo/stageA_quickdraw/` |
| 학습 파라미터(`QD_EPOCHS`, `MODEL_ARCH` …) | 없음 — `FORCE_RETRAIN=True`로 재학습 |

Config를 고친 뒤에는 **커널 재시작 → Run All**.

## 1. 환경 세팅

In [ ]:
# 필요 패키지 설치 (JupyterHub/VSCode 공통). 이미 설치돼 있으면 그대로 통과.
%pip install -q "ultralytics>=8.3.0" opencv-python-headless pillow pyyaml tqdm matplotlib pandas

In [ ]:
# Stage A는 zip·HTP 관련 의존이 없다 — Quick Draw 다운로드와 합성만 필요하다.
import os

# ── GPU 선택: 물리 GPU 6번만 사용 (공용 서버 지정) ────────────────
#   ⚠️ import torch 보다 반드시 먼저 실행돼야 적용된다. 이미 torch가 로드된
#      뒤면 무시되므로 커널 재시작 후 이 셀부터 다시 실행할 것.
#   ⚠️ 값은 문자열이어야 한다. CUDA_VISIBLE_DEVICES="6" 을 걸면 torch에는
#      그 GPU가 유일한 장치(cuda:0)로 보이므로 Config의 DEVICE 는 6이 아니라
#      0 으로 두어야 한다(그대로 두면 됨).
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

import sys, json, random, shutil, math
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.auto import tqdm

try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    TORCH_VER = torch.__version__
except ModuleNotFoundError:
    torch = None; CUDA_AVAILABLE = False; TORCH_VER = "(미설치)"

print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("PyTorch     :", TORCH_VER)
print("CUDA 사용가능:", CUDA_AVAILABLE)
if CUDA_AVAILABLE:
    print("GPU         :", torch.cuda.get_device_name(0), f"(x{torch.cuda.device_count()})")

In [ ]:
import io, urllib.request
from urllib.parse import quote
from PIL import Image, ImageDraw, ImageChops

In [ ]:
# ── 한글 폰트 설정 (matplotlib 라벨 깨짐 방지) ──────────────────────
# Windows/Linux/macOS 공통. 이 셀을 먼저 실행하면 이후 모든 시각화에 한글이 적용된다.
# JupyterHub(리눅스, sudo 없음)처럼 시스템에 한글 폰트가 없으면 NanumGothic을
# 유저 폴더에 자동 내려받아 등록한다 — apt/root 권한이 필요 없다.
import urllib.request
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 유저 홈에 폰트를 캐시(쓰기 가능·재실행 시 재사용). 실패하면 현재 폴더로 대체.
_FONT_URL = ("https://github.com/google/fonts/raw/main/ofl/nanumgothic/"
             "NanumGothic-Regular.ttf")
try:
    _FONT_CACHE = Path.home() / ".fonts"
    _FONT_CACHE.mkdir(parents=True, exist_ok=True)
except Exception:
    _FONT_CACHE = Path("fonts"); _FONT_CACHE.mkdir(exist_ok=True)
_FONT_DL = _FONT_CACHE / "NanumGothic-Regular.ttf"


def _download_korean_font():
    # 이미 받아둔 게 있으면 재사용, 없으면 내려받는다. 반환: 성공 시 파일 경로.
    if _FONT_DL.exists() and _FONT_DL.stat().st_size > 100_000:
        return _FONT_DL
    try:
        req = urllib.request.Request(_FONT_URL, headers={"User-Agent": "htp/1.0"})
        with urllib.request.urlopen(req, timeout=120) as r:
            _FONT_DL.write_bytes(r.read())
        print(f"NanumGothic 다운로드: {_FONT_DL}")
        return _FONT_DL
    except Exception as e:
        print(f"⚠️ 폰트 다운로드 실패({type(e).__name__}: {e}) — 네트워크/프록시를 확인하세요.")
        return None


def _apply_font(fp):
    font_manager.fontManager.addfont(str(fp))
    name = font_manager.FontProperties(fname=str(fp)).get_name()
    plt.rcParams["font.family"] = name
    plt.rcParams["axes.unicode_minus"] = False
    return name


def setup_korean_font():
    candidates = [
        r"C:\Windows\Fonts\malgun.ttf",        # Windows 맑은 고딕
        r"C:\Windows\Fonts\NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",          # Linux (apt)
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/System/Library/Fonts/AppleSDGothicNeo.ttc",               # macOS
        "/Library/Fonts/AppleGothic.ttf",
        str(_FONT_DL),                          # 이전 실행에서 받아둔 것
    ]
    for fp in candidates:
        if Path(fp).exists():
            name = _apply_font(fp)
            print(f"한글 폰트: {name}  ({fp})")
            return fp, name
    # 설치된 폰트 이름으로 시도(파일 경로로 못 찾은 경우)
    for name in ["Malgun Gothic", "NanumGothic", "AppleGothic", "Noto Sans CJK KR"]:
        if any(f.name == name for f in font_manager.fontManager.ttflist):
            plt.rcParams["font.family"] = name
            plt.rcParams["axes.unicode_minus"] = False
            print(f"한글 폰트(이름): {name}")
            return None, name
    # 마지막 수단: 유저 폴더로 다운로드해 등록 (JupyterHub 등 root 없는 환경)
    fp = _download_korean_font()
    if fp is not None:
        name = _apply_font(fp)
        print(f"한글 폰트(다운로드): {name}  ({fp})")
        return str(fp), name
    print("⚠️ 한글 폰트를 찾지 못했습니다. 라벨이 깨질 수 있습니다.")
    return None, None

KOREAN_FONT_PATH, KOREAN_FONT_NAME = setup_korean_font()

## 2. Config  ⚙️

**환경을 바꿀 때는 이 셀만 수정한다.** 기본값은 스모크 테스트용이다.

In [ ]:
# ═══════════════════════ Config (Stage A) ═══════════════════════
# ⚠️ 절대경로 필수: 상대경로를 쓰면 Ultralytics가 project= 인자를
#    ~/AppData/Roaming/Ultralytics/settings.json 의 runs_dir(다른 프로젝트 설정값일 수 있음)와
#    합쳐 엉뚱한 위치(runs/detect/htp_yolo/runs/...)에 저장해버린다. (실제 발생한 함정)
OUTPUT_ROOT = Path("htp_yolo").resolve()         # 산출물 루트 (htp_finetune.ipynb와 공유)
QD_RAW_DIR  = OUTPUT_ROOT / "quickdraw_raw"      # Quick Draw 원본 ndjson 캐시
QD_DS_DIR   = OUTPUT_ROOT / "stageA_quickdraw"   # 합성 데이터셋
RUNS_DIR    = OUTPUT_ROOT / "runs"
RUN_NAME    = "stageA_quickdraw_v2"              # 학습 로그 폴더명 (33종판과 분리)

# ---- 산출물 이름 ----
#   클래스를 확장해 새 모델을 만들 때는 반드시 이 이름을 바꾼다.
#   기존 sketch_base.pt는 현재 htp_best.pt의 시작 가중치라 덮어쓰면 계보가 끊긴다.
#   예) 확장판: SKETCH_MODEL_NAME = "sketch_base_v2.pt", RUN_NAME = "stageA_quickdraw_v2"
SKETCH_MODEL_NAME = "sketch_base_v2.pt"   # 100클래스 확장판. 기존 33종 sketch_base.pt는 보존됨
SKETCH_BASE       = OUTPUT_ROOT / SKETCH_MODEL_NAME

FORCE_RETRAIN = False   # True면 SKETCH_BASE가 이미 있어도 다시 학습해 덮어쓴다

# ---- 학습 하이퍼파라미터 ----
MODEL_ARCH = "yolo11s.pt"   # 스모크: yolo11n / 본학습: yolo11s (또는 yolo11m)
IMG_SIZE   = 960            # 스모크: 640 / 본학습: 960~1280
BATCH      = 16
WORKERS    = 8
DEVICE     = 0 if CUDA_AVAILABLE else "cpu"   # 오버라이드: "cpu" / 0 / "0,1"
SEED       = 42

# ---- 추가 튜닝: Ultralytics train() 인자를 자유롭게 전달 ----
# 예) {"cos_lr": True, "lr0": 0.001, "optimizer": "AdamW", "mosaic": 0.5, "close_mosaic": 10}
# freeze는 넣지 않는다 — Stage A는 COCO에서 처음부터 학습하는 단계라 백본을 얼릴 이유가 없다
# (freeze는 htp_finetune.ipynb의 파인튜닝 전용).
STAGE_A_TRAIN_KW = {}

# ---- 합성 데이터셋 ----
QD_EPOCHS          = 20      # 스모크: 10 / 본학습: 30~50
QD_PER_CATEGORY    = 800     # 카테고리당 내려받을 그림 수 (스모크 800 / 본학습 3000+)
QD_TRAIN_IMAGES    = 5000    # 합성 학습 이미지 수 (클래스 100종 → 클래스당 ~300 인스턴스). 스모크는 1500
QD_VAL_IMAGES      = 200
QD_CANVAS          = 640     # 합성 캔버스 해상도
QD_OBJ_PER_IMAGE   = (3, 9)  # 이미지당 객체 수 범위
QD_ONLY_RECOGNIZED = True    # Quick Draw가 '인식 성공'으로 표시한 그림만 사용(품질 필터)
QD_BASE_URL = "https://storage.googleapis.com/quickdraw_dataset/full/simplified/"

# Quick Draw 카테고리 100종 — 만 12세 이하 아동 자유화 기준으로 선정, URL 존재 전수 검증 완료.
#   앞 33종: HTP 요소 대응(sketch_base.pt와 동일). 그 뒤 67종: 동물·탈것·음식·자연·사물 확장.
#   HTP에 없는 요소(window/roof/chimney/smoke/hair/sneaker)는 Quick Draw에도 없어 제외.
# ⚠️ 카테고리명은 Quick Draw 공식 표기와 정확히 일치해야 한다(틀리면 다운로드 404).
# ⚠️ 이 목록을 바꾸면 htp_yolo/stageA_quickdraw/ 를 지워야 반영된다(이미지 수가 같으면 캐시가 그대로 쓰임).
QD_CATEGORIES = [
    "house", "tree", "door", "fence", "flower", "grass",
    "bush", "sun", "cloud", "moon", "star", "bird",
    "squirrel", "mountain", "face", "eye", "nose", "mouth",
    "ear", "arm", "hand", "leg", "foot", "shoe",
    "sock", "t-shirt", "pants", "swing set", "ladder", "stairs",
    "river", "pool", "garden", "cat", "dog", "rabbit",
    "bear", "lion", "tiger", "elephant", "giraffe", "monkey",
    "pig", "cow", "horse", "sheep", "duck", "penguin",
    "owl", "fish", "butterfly", "bee", "spider", "frog",
    "snake", "car", "bus", "truck", "train", "airplane",
    "helicopter", "bicycle", "school bus", "firetruck", "apple", "banana",
    "strawberry", "watermelon", "carrot", "ice cream", "cake", "pizza",
    "hamburger", "cookie", "donut", "lollipop", "rainbow", "snowman",
    "snowflake", "lightning", "palm tree", "leaf", "chair", "table",
    "bed", "clock", "book", "cup", "umbrella", "pencil",
    "hat", "castle", "bridge", "smiley face", "crown", "soccer ball",
    "basketball", "skateboard", "guitar", "piano",
]

random.seed(SEED); np.random.seed(SEED)
for d in (OUTPUT_ROOT, QD_RAW_DIR, RUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# 클래스 수 대비 합성 이미지가 너무 적으면 희귀 클래스가 학습되지 않는다.
# 이미지당 평균 객체 수 기준으로 '클래스당 기대 인스턴스'를 추정해 경고한다.
_avg_obj = sum(QD_OBJ_PER_IMAGE) / 2
_per_cls = QD_TRAIN_IMAGES * _avg_obj / max(len(QD_CATEGORIES), 1)

print("OUTPUT_ROOT :", OUTPUT_ROOT)
print("산출물      :", SKETCH_BASE, "(이미 있음)" if SKETCH_BASE.exists() else "(신규)")
print(f"arch={MODEL_ARCH} imgsz={IMG_SIZE} batch={BATCH} device={DEVICE} epochs={QD_EPOCHS}")
print(f"카테고리 {len(QD_CATEGORIES)}개 · 합성 train {QD_TRAIN_IMAGES}장 / val {QD_VAL_IMAGES}장")
print(f"클래스당 기대 인스턴스 ≈ {_per_cls:.0f}개", end="  ")
print("⚠️ 200 미만 — QD_TRAIN_IMAGES를 올리세요" if _per_cls < 200 else "✓")
if SKETCH_BASE.exists() and not FORCE_RETRAIN:
    print("→ 학습 셀은 기존 가중치를 재사용한다(다시 학습하려면 FORCE_RETRAIN=True).")

### Quick Draw ↔ HTP 클래스 대응 (참고)

Stage A와 Stage B는 **클래스 수가 다르므로 검출 헤드는 재초기화**된다.
전이되는 가치는 **백본/넥이 학습한 선화(line-art) 특징**이며, 이는 의도된 설계다.
아래 표는 "Stage A가 HTP에 얼마나 관련된 것을 배우는가"를 보여주는 참고자료일 뿐,
학습에는 쓰이지 않는다.

In [ ]:
# ── Quick Draw → HTP 대응 참고표 ──────────────────────────────────
QD_TO_HTP = {
    "house": "집전체", "tree": "나무전체", "door": "문", "fence": "울타리",
    "flower": "꽃", "grass": "잔디", "bush": "잔디(유사)", "sun": "태양",
    "cloud": "구름", "moon": "달", "star": "별", "bird": "새",
    "squirrel": "다람쥐", "mountain": "산", "face": "얼굴", "eye": "눈",
    "nose": "코", "mouth": "입", "ear": "귀", "arm": "팔", "hand": "손",
    "leg": "다리", "foot": "발", "shoe": "남자구두/여자구두", "sock": "발(유사)",
    "t-shirt": "상체", "pants": "다리(유사)", "swing set": "그네",
    "pool": "연못", "river": "길(유사)", "ladder": "(보조)",
    "stairs": "(보조)", "garden": "(보조)",
}
display(pd.DataFrame([{"QuickDraw": c, "HTP 대응": QD_TO_HTP.get(c, "(보조)")}
                      for c in QD_CATEGORIES]))

# HTP 클래스 목록은 htp_finetune.ipynb가 만든다. 있으면 커버리지를 계산하고,
# 없으면 조용히 건너뛴다 — 이 노트북은 HTP 산출물에 의존하지 않는다.
_cls_json = OUTPUT_ROOT / "classes.json"
if _cls_json.exists():
    _htp_names = json.loads(_cls_json.read_text(encoding="utf-8"))["names"]
    _covered = {v for v in QD_TO_HTP.values() if v in _htp_names}
    print(f"HTP {len(_htp_names)}개 중 Quick Draw로 직접 대응되는 클래스: {len(_covered)}개")
    print("대응 없는 HTP 클래스(예):",
          ", ".join(sorted(set(_htp_names) - _covered)[:15]), "...")
else:
    print("classes.json 없음 — 커버리지 계산 생략 (htp_finetune.ipynb 실행 시 생성됨)")

## 3. 합성 데이터셋 생성  🎨

### 핵심 아이디어
Quick Draw 원본은 **한 그림에 객체 1개**이고 좌표가 0–255를 꽉 채우도록 정규화되어 있어,
그대로 쓰면 bbox가 항상 이미지 전체가 된다(퇴화 과제). 따라서:

1. 카테고리별 스케치를 스트리밍으로 내려받는다(필요한 만큼만 읽고 조기 종료 → 전체 다운로드 불필요).
2. 스케치를 **랜덤 크기로 렌더링**해 빈 캔버스의 **랜덤 위치에 합성**한다.
3. 배치를 우리가 하므로 **bbox를 정확히 계산**할 수 있다(잉크 픽셀 min/max).
4. 연필 느낌에 가깝도록 선 두께·농도·약간의 회전을 무작위화해 도메인 격차를 줄인다.

In [ ]:
# ── Quick Draw 다운로드 (필요한 줄만 스트리밍, 캐시) ──────────────
def qd_download(category: str, n: int, dest_dir: Path) -> Path:
    dest = dest_dir / f"{category.replace(' ', '_')}.ndjson"
    if dest.exists():
        with dest.open(encoding="utf-8") as f:
            if sum(1 for _ in f) >= n:
                return dest   # 캐시 충분
    url = QD_BASE_URL + quote(category) + ".ndjson"
    lines = []
    req = urllib.request.Request(url, headers={"User-Agent": "htp-baseline/1.0"})
    with urllib.request.urlopen(req, timeout=180) as r:
        for raw in r:                      # ndjson: 줄 단위 스트리밍
            lines.append(raw.decode("utf-8"))
            if len(lines) >= n:
                break                      # 필요한 만큼만 받고 조기 종료
    dest.write_text("".join(lines), encoding="utf-8")
    return dest

def qd_load(category: str, n: int) -> list:
    # 스트로크 리스트 반환. 각 스트로크는 [[x...], [y...]] (0~255 좌표계)
    p = QD_RAW_DIR / f"{category.replace(' ', '_')}.ndjson"
    out = []
    with p.open(encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            if QD_ONLY_RECOGNIZED and not d.get("recognized", True):
                continue
            out.append(d["drawing"])
            if len(out) >= n:
                break
    return out

QD_POOL = {}
for cat in tqdm(QD_CATEGORIES, desc="Quick Draw 다운로드"):
    qd_download(cat, QD_PER_CATEGORY, QD_RAW_DIR)
    QD_POOL[cat] = qd_load(cat, QD_PER_CATEGORY)

QD_NAMES   = sorted(QD_POOL)                       # Stage A 클래스(결정론적 정렬)
QD_NAME_ID = {c: i for i, c in enumerate(QD_NAMES)}
print(f"\n카테고리 {len(QD_NAMES)}개, 그림 총 {sum(len(v) for v in QD_POOL.values()):,}개 확보")
print("카테고리별 개수 예시:", {c: len(QD_POOL[c]) for c in QD_NAMES[:5]})

In [ ]:
# ── 스케치 렌더링 + 합성 캔버스 생성 ──────────────────────────────
def render_sketch(strokes, size: int, rng: random.Random):
    # 스케치를 size x size 패치(L 모드, 흰 배경)에 렌더링.
    # 반환: (패치, 패치 내 tight bbox (x1,y1,x2,y2))
    pts = [(x, y) for s in strokes for x, y in zip(s[0], s[1])]
    if len(pts) < 2:
        return None, None
    xs = [p[0] for p in pts]; ys = [p[1] for p in pts]
    x0, x1 = min(xs), max(xs); y0, y1 = min(ys), max(ys)
    sw = max(1, int(round(size / rng.choice([70, 90, 120]))))   # 선 두께 무작위
    pad = sw + 2
    span = max(x1 - x0, y1 - y0, 1)
    scale = (size - 2 * pad) / span

    patch = Image.new("L", (size, size), 255)
    dr = ImageDraw.Draw(patch)
    ink = rng.randint(40, 110)          # 연필 농도 무작위
    def tx(x, y):
        return (pad + (x - x0) * scale, pad + (y - y0) * scale)
    for s in strokes:
        pl = [tx(x, y) for x, y in zip(s[0], s[1])]
        if len(pl) == 1:
            dr.ellipse([pl[0][0]-sw, pl[0][1]-sw, pl[0][0]+sw, pl[0][1]+sw], fill=ink)
        else:
            dr.line(pl, fill=ink, width=sw, joint="curve")
    if rng.random() < 0.5:              # 약간의 회전
        patch = patch.rotate(rng.uniform(-8, 8), resample=Image.BILINEAR, fillcolor=255)

    # tight bbox는 렌더링이 모두 끝난 뒤 '실제 잉크 픽셀'에서 계산한다.
    # (좌표 계산으로 구하면 회전이 적용됐을 때 잉크가 박스 밖으로 삐져나간다)
    arr = np.asarray(patch)
    ink_px = np.argwhere(arr < 250)     # 배경 255, 안티에일리어싱 여유
    if ink_px.size == 0:
        return None, None
    (iy1, ix1), (iy2, ix2) = ink_px.min(0), ink_px.max(0)
    bb = (float(ix1), float(iy1), float(ix2 + 1), float(iy2 + 1))
    return patch, bb

def make_canvas(rng: random.Random):
    # Quick Draw 스케치를 랜덤 배치한 합성 장면 1장 생성.
    C = QD_CANVAS
    canvas = Image.new("L", (C, C), rng.randint(248, 255))   # 살짝 종이톤
    boxes = []
    for _ in range(rng.randint(*QD_OBJ_PER_IMAGE)):
        cat = rng.choice(QD_NAMES)
        if not QD_POOL[cat]:
            continue
        strokes = rng.choice(QD_POOL[cat])
        s = rng.randint(int(C * 0.12), int(C * 0.42))
        patch, bb = render_sketch(strokes, s, rng)
        if patch is None:
            continue
        px, py = rng.randint(0, C - s), rng.randint(0, C - s)
        region = canvas.crop((px, py, px + s, py + s))
        canvas.paste(ImageChops.darker(region, patch), (px, py))  # 겹침 시 진한 쪽 유지
        x1, y1, x2, y2 = bb[0] + px, bb[1] + py, bb[2] + px, bb[3] + py
        if x2 - x1 >= 4 and y2 - y1 >= 4:
            boxes.append((QD_NAME_ID[cat], x1, y1, x2, y2))
    return canvas.convert("RGB"), boxes

def write_yolo_split(out_dir: Path, n_images: int, seed: int, desc: str):
    img_d, lbl_d = out_dir / "images", out_dir / "labels"
    img_d.mkdir(parents=True, exist_ok=True); lbl_d.mkdir(parents=True, exist_ok=True)
    rng = random.Random(seed)
    C = QD_CANVAS
    for i in tqdm(range(n_images), desc=desc, leave=False):
        img, boxes = make_canvas(rng)
        img.save(img_d / f"qd_{i:06d}.jpg", quality=92)
        lines = [f"{c} {((x1+x2)/2)/C:.6f} {((y1+y2)/2)/C:.6f} {(x2-x1)/C:.6f} {(y2-y1)/C:.6f}"
                 for c, x1, y1, x2, y2 in boxes]
        (lbl_d / f"qd_{i:06d}.txt").write_text("\n".join(lines), encoding="utf-8")

# 캐시 판정은 이미지 수 '이상'으로 본다. 카테고리를 바꿨는데 이미지 수가 같으면
# 캐시가 그대로 쓰이므로, 클래스를 바꿨다면 stageA_quickdraw/ 를 직접 지워야 한다.
_train_img_dir = QD_DS_DIR / "train" / "images"
_cached = _train_img_dir.exists() and len(list(_train_img_dir.glob("*.jpg"))) >= QD_TRAIN_IMAGES
if _cached:
    print("합성 데이터셋 캐시 사용:", QD_DS_DIR.resolve())
    print("  ⚠️ QD_CATEGORIES를 바꿨다면 이 폴더를 지우고 다시 실행해야 반영된다.")
else:
    shutil.rmtree(QD_DS_DIR, ignore_errors=True)
    write_yolo_split(QD_DS_DIR / "train", QD_TRAIN_IMAGES, SEED,     "합성 train")
    write_yolo_split(QD_DS_DIR / "val",   QD_VAL_IMAGES,   SEED + 1, "합성 val")
    print("생성 완료:", QD_DS_DIR.resolve())

qd_yaml = QD_DS_DIR / "data.yaml"
qd_yaml.write_text(
    "path: " + str(QD_DS_DIR.resolve()) + "\ntrain: train/images\nval: val/images\n"
    + "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(QD_NAMES)),
    encoding="utf-8")
print("data.yaml:", qd_yaml.resolve(), "| 클래스", len(QD_NAMES), "개")

In [ ]:
# ── 합성 결과 검증: bbox를 이미지 위에 그려 확인 ──────────────────
smp = sorted((QD_DS_DIR / "train" / "images").glob("*.jpg"))[:3]
fig, axes = plt.subplots(1, len(smp), figsize=(6 * len(smp), 6))
if len(smp) == 1: axes = [axes]
for ax, ip in zip(axes, smp):
    im = Image.open(ip); W, H = im.size
    ax.imshow(im, cmap="gray"); ax.axis("off"); ax.set_title(ip.name, fontsize=9)
    for line in (ip.parent.parent / "labels" / (ip.stem + ".txt")).read_text().splitlines():
        c, xc, yc, w, h = line.split()
        c = int(c); xc, yc, w, h = float(xc)*W, float(yc)*H, float(w)*W, float(h)*H
        ax.add_patch(patches.Rectangle((xc-w/2, yc-h/2), w, h,
                                       fill=False, edgecolor="red", linewidth=1.2))
        ax.text(xc-w/2, max(yc-h/2-3, 0), QD_NAMES[c], color="white", fontsize=7,
                bbox=dict(facecolor="red", alpha=0.6, pad=0.5, edgecolor="none"))
plt.tight_layout(); plt.show()

## 4. 학습 — 범용 손그림 탐지기

COCO 사전학습 가중치에서 시작해 합성 Quick Draw 장면으로 학습한다.
산출물은 그 자체로 최종 납품 대상(**산출물 1**)이면서, `htp_finetune.ipynb`의 시작 가중치가 된다.

In [ ]:
from ultralytics import YOLO

if SKETCH_BASE.exists() and not FORCE_RETRAIN:
    print("기존 가중치 재사용:", SKETCH_BASE.resolve())
    print("  다시 학습하려면 Config에서 FORCE_RETRAIN=True (또는 SKETCH_MODEL_NAME을 새 이름으로)")
else:
    m = YOLO(MODEL_ARCH)                      # COCO 사전학습에서 시작
    m.train(data=str(qd_yaml.resolve()),
            epochs=QD_EPOCHS, imgsz=IMG_SIZE, batch=BATCH, workers=WORKERS,
            device=DEVICE, seed=SEED, project=str(RUNS_DIR), name=RUN_NAME,
            exist_ok=True, verbose=True, **STAGE_A_TRAIN_KW)
    shutil.copy(RUNS_DIR / RUN_NAME / "weights" / "best.pt", SKETCH_BASE)
    print("\n학습 완료 →", SKETCH_BASE.resolve())

## 5. 평가

범용 모델은 HTP 파인튜닝의 중간 산출물이 아니라 **독립 산출물**이므로 자체 지표를 따로 남긴다.
합성 데이터셋에는 test 분할이 없어 `val`(학습에 미사용)로 평가한다.

In [ ]:
# ── val 분할 평가 + 클래스별 지표 ─────────────────────────────────
_model = YOLO(str(SKETCH_BASE))
_r = _model.val(data=str(qd_yaml.resolve()), split="val", imgsz=IMG_SIZE,
                batch=BATCH, device=DEVICE, project=str(RUNS_DIR),
                name=f"eval_{RUN_NAME}", exist_ok=True, verbose=False)

sketch_metric = {"mAP@50": _r.box.map50, "mAP@50-95": _r.box.map,
                 "precision": _r.box.mp, "recall": _r.box.mr}
display(pd.DataFrame({SKETCH_MODEL_NAME: sketch_metric}).T.style.format("{:.4f}"))

per_cls = pd.DataFrame([
    {"클래스": QD_NAMES[c], "precision": _r.box.p[i], "recall": _r.box.r[i],
     "mAP@50": _r.box.ap50[i], "mAP@50-95": _r.box.ap[i]}
    for i, c in enumerate(_r.box.ap_class_index)
]).sort_values("mAP@50-95", ascending=False)
display(per_cls.style.format({c: "{:.4f}" for c in per_cls.columns if c != "클래스"}))
print("\n성능 하위 10개 클래스 (확장 시 데이터 보강 우선순위):")
display(per_cls.tail(10))

In [ ]:
# ── 실험 기록용 요약 (TSV 한 줄 → 스프레드시트) ───────────────────
import datetime

_csv = RUNS_DIR / RUN_NAME / "results.csv"
_sec = round(float(pd.read_csv(_csv).iloc[-1]["time"]), 1) if _csv.exists() else None

row = {
    "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
    "model": SKETCH_MODEL_NAME,
    "arch": MODEL_ARCH, "imgsz": IMG_SIZE, "batch": BATCH,
    "n_classes": len(QD_NAMES), "per_category": QD_PER_CATEGORY,
    "train_images": QD_TRAIN_IMAGES, "epochs": QD_EPOCHS,
    "train_sec": _sec,
    "mAP50": round(sketch_metric["mAP@50"], 4),
    "mAP50-95": round(sketch_metric["mAP@50-95"], 4),
    "precision": round(sketch_metric["precision"], 4),
    "recall": round(sketch_metric["recall"], 4),
}
log_row = pd.DataFrame([row])
print("=== 이번 실험 요약 (Stage A) ===")
display(log_row.T.rename(columns={0: "값"}))
print("\n=== 스프레드시트 붙여넣기용 (탭 구분) ===")
print("\t".join(log_row.columns))
print("\t".join(str(v) for v in log_row.iloc[0]))

In [ ]:
# ── 예측 결과 시각화 (val 분할) ───────────────────────────────────
val_imgs = sorted((QD_DS_DIR / "val" / "images").glob("*.jpg"))[:4]
preds = _model.predict([str(p) for p in val_imgs], imgsz=IMG_SIZE, conf=0.25,
                       device=DEVICE, verbose=False)

pred_dir = OUTPUT_ROOT / "predictions"; pred_dir.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, len(preds), figsize=(6 * len(preds), 6))
if len(preds) == 1: axes = [axes]
for ax, p, ip in zip(axes, preds, val_imgs):
    ax.imshow(Image.open(ip).convert("RGB")); ax.axis("off"); ax.set_title(ip.stem, fontsize=8)
    b = p.boxes
    for (x1, y1, x2, y2), cls, conf in zip(b.xyxy.tolist(), b.cls.tolist(), b.conf.tolist()):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       fill=False, edgecolor="red", linewidth=1.2))
        ax.text(x1, max(y1 - 3, 0), f"{QD_NAMES[int(cls)]} {conf:.2f}",
                color="white", fontsize=7,
                bbox=dict(facecolor="red", alpha=0.6, pad=0.5, edgecolor="none"))
plt.tight_layout()
_out = pred_dir / f"{Path(SKETCH_MODEL_NAME).stem}_predictions.png"
fig.savefig(_out, dpi=120, bbox_inches="tight")
plt.show()
print("예측 이미지 저장:", _out.resolve())

## 정리

### 산출물
| 경로 | 내용 |
|---|---|
| `htp_yolo/<SKETCH_MODEL_NAME>` | **범용 손그림 탐지 모델 (산출물 1)** |
| `htp_yolo/stageA_quickdraw/` | 합성 데이터셋 + `data.yaml` |
| `htp_yolo/quickdraw_raw/` | Quick Draw 원본 ndjson 캐시 |
| `htp_yolo/runs/<RUN_NAME>/` | 학습 로그·곡선·`weights/best.pt` |
| `htp_yolo/runs/eval_<RUN_NAME>/` | val 평가 결과 |

### 다음 단계
HTP 전용 모델이 필요하면 [htp_finetune.ipynb](htp_finetune.ipynb)를 실행한다.
그 노트북 Config의 `SKETCH_MODEL_NAME`이 여기서 만든 파일을 가리키는지 확인할 것.

### 클래스 확장 체크리스트
1. `QD_CATEGORIES` 확장 (공식 표기 일치 확인 — 틀리면 404)
2. `QD_TRAIN_IMAGES` 상향 (Config 셀의 "클래스당 기대 인스턴스" 경고 참고)
3. `SKETCH_MODEL_NAME`·`RUN_NAME`을 새 이름으로 — 기존 가중치 보존
4. `htp_yolo/stageA_quickdraw/` 삭제
5. 커널 재시작 → Run All
6. `model_download.ipynb`의 `MODEL_FILES`에 새 파일 추가